# NEXORA বাংলা ↔ ইংরেজি অনুবাদ পরীক্ষা

এই Notebook কেবল synthetic test corpus ব্যবহার করে। Buyer/Seller/Porter/Admin-এর ব্যক্তিগত Chat কপি করবেন না। প্রথমে Runtime → Change runtime type থেকে CPU/GPU নির্বাচন করতে পারেন; বিনামূল্যের GPU নিশ্চিত নয়। কোনো paid runtime বেছে নেবেন না।

**গুরুত্বপূর্ণ:** IndicTrans2-এর দুইটি official checkpoint gated। Hugging Face account-এ model access terms গ্রহণ করতে হবে। Read-only token কেবল নিচের secure prompt-এ দিন; Notebook-এ token লিখবেন না বা GitHub-এ commit করবেন না।

## ধাপ ১ — Repository ও corpus প্রস্তুত

In [ ]:
import os, pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/nexora-platform1')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--depth','1','--branch','nexora-development','https://github.com/shaifrony/nexora-platform1.git',str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Branch checkout:', subprocess.check_output(['git','branch','--show-current'], text=True).strip())
print('Corpus exists:', (REPO_DIR/'docs/NEXORA_BN_EN_PILOT_CORPUS.md').exists())
print('Runner exists:', (REPO_DIR/'scripts/translation_pilot/run_bn_en_pilot.py').exists())

## ধাপ ২ — Argos dependencies
এই ধাপটি দুই দিকের Argos 1.9 প্যাকেজ চালানোর চেষ্টা করবে। প্যাকেজের commercial license এখনো NEXORA production-এর জন্য যাচাই করা হয়নি।

In [ ]:
%pip -q install --upgrade pip
%pip -q install argostranslate huggingface_hub 'transformers>=4.51,<5' 'numpy>=2.1' sentencepiece sacremoses accelerate indictranstoolkit

## ধাপ ৩ — Hugging Face access ও checkpoint pinning
দুইটি মডেলের জন্য আপনার Hugging Face account-এ আগে access condition গ্রহণ করুন। নিচে Read token secure prompt-এ দিন। Token output-এ print হবে না এবং Notebook cell-এ সংরক্ষণ করা হবে না।

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi, login
token = getpass('Hugging Face READ token (input hidden): ')
login(token=token, add_to_git_credential=False)
api = HfApi()
model_ids = {
  'HF_REVISION_EN_BN': 'ai4bharat/indictrans2-en-indic-dist-200M',
  'HF_REVISION_BN_EN': 'ai4bharat/indictrans2-indic-en-dist-200M',
}
for env_name, model_id in model_ids.items():
    info = api.model_info(model_id)
    os.environ[env_name] = info.sha
    print(env_name, model_id, info.sha)
del token
print('Checkpoint revisions pinned for this runtime only.')

## ধাপ ৪ — একই corpus দিয়ে উভয় মডেল চালানো
এতে প্রথমবার মডেল ডাউনলোড হতে পারে এবং কয়েক GB ডেটা/বেশি RAM লাগতে পারে। Runtime disconnect হলে পুনরায় চালাতে হতে পারে।

In [ ]:
import subprocess, sys
cmd = [sys.executable, 'scripts/translation_pilot/run_bn_en_pilot.py', '--backend', 'both', '--direction', 'both']
result = subprocess.run(cmd, text=True)
print('Runner exit code:', result.returncode)
if result.returncode != 0:
    print('One or more backends may have failed. Inspect error output; do not treat missing rows as test results.')

## ধাপ ৫ — ফলাফল CSV দেখুন ও ডাউনলোড করুন
CSV-তে raw translation output ও latency থাকবে। এটি নিজে থেকে model quality score বা winner নির্ধারণ করবে না। দুইজন bilingual reviewer দিয়ে reference ও output যাচাই করতে হবে।

In [ ]:
from pathlib import Path
from IPython.display import display
files = sorted(Path('artifacts/translation-pilot').glob('bn-en-results-*.csv'))
if not files:
    print('No result CSV found. Runner did not produce results.')
else:
    latest = files[-1]
    print('Result file:', latest)
    import pandas as pd
    display(pd.read_csv(latest).head(12))
    from google.colab import files
    files.download(str(latest))

## নিরাপত্তা ও সীমাবদ্ধতা
- শুধু repository-র synthetic corpus ব্যবহার করুন; live chat, phone, address, identity বা order data নয়।
- Colab runtime অস্থায়ী; session শেষে local files মুছে যেতে পারে। CSV নিজের নিরাপদ জায়গায় ডাউনলোড করুন।
- Colab free GPU/RAM/time guaranteed নয়; Google-এর resource limits dynamic। কোনো paid upgrade প্রয়োজন হলে থামুন—অনুমতি ছাড়া কিনবেন না।
- এই পরীক্ষার ফলাফল NEXORA production translation enable করে না। Model quality, privacy, cost, latency এবং human review আলাদাভাবে অনুমোদন করতে হবে।